# 四分类证据一致性评测 Notebook

这个 Notebook 用于替代旧版“是否被证据支持”的二分类评测流程，升级为四分类版本，并且尽量保持自包含、可迁移、可扩展。

评测对象是三元组：`问题 + 证据 + 回答`。

新的主标签体系为：

- `supported`
- `contradicted`
- `insufficient`
- `corrected-premise`

支持的次标签为：

- `hallucinated-detail`
- `overclaim`
- `missed-correction`
- `safe-but-unhelpful`
- `partial-support`
- `negation-failure`
- `irrelevant-extension`

为什么四分类比旧版“支持 / 不支持”更稳：

1. 二分类很容易把“提到了相关实体”误判成“被证据支持”，但相关不等于支持。
2. 二分类无法区分“证据明确反驳”和“证据不足以支持”，这两类错误的风险不同。
3. 当问题本身带有错误前提时，回答如果成功纠正了前提，应该得到单独奖励，因此需要 `corrected-premise`。
4. 四分类更适合分析模型到底是在变保守、变准确，还是只是减少输出。

本 Notebook 默认提供一个“规则演示模式”，因此不下载模型也能直接运行 demo；同时也保留了一个可切换到本地 `transformers` chat 模型的 `call_judge_model()` 实现。如果你已有自己的生成函数，也可以按后文说明替换。


## 第 1 部分：导入依赖

这一块只负责导入后续需要使用的标准库和第三方库。

输入：无。
输出：当前 Notebook 环境可直接使用的依赖。

如果你的环境还没有安装依赖，可以先执行：

```bash
pip install pandas transformers torch
```

如果你只打算把 `call_judge_model()` 替换成已有接口，可以不使用 `transformers` 模型加载部分，但保留这里的导入通常更方便后续扩展。


In [ ]:
import json
import re
from typing import Any, Dict, List, Optional

import pandas as pd

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
except Exception:
    torch = None
    AutoModelForCausalLM = None
    AutoTokenizer = None


## 第 2 部分：标签集合、评分规则与辅助判断函数

这一块定义四分类主标签、次标签，以及后续汇总统计时要用到的简单评分规则。

输入：主标签字符串。
输出：标签合法性判断、评分、是否安全、是否高质量等辅助结果。

这里的默认评分设计为：

- `supported = 1.0`
- `corrected-premise = 1.0`
- `insufficient = 0.5`
- `contradicted = 0.0`

你可以按自己的项目要求调整分数，但推荐保留 `corrected-premise` 与 `supported` 分开统计。


In [ ]:
MAIN_LABELS = {
    "supported",
    "contradicted",
    "insufficient",
    "corrected-premise",
}

SECONDARY_LABELS = {
    "hallucinated-detail",
    "overclaim",
    "missed-correction",
    "safe-but-unhelpful",
    "partial-support",
    "negation-failure",
    "irrelevant-extension",
}

MAIN_LABEL_SCORE = {
    "supported": 1.0,
    "corrected-premise": 1.0,
    "insufficient": 0.5,
    "contradicted": 0.0,
}

def is_safe_label(main_label: str) -> bool:
    return main_label in {"supported", "corrected-premise", "insufficient"}

def is_good_label(main_label: str) -> bool:
    return main_label in {"supported", "corrected-premise"}

def validate_main_label(main_label: str) -> str:
    if main_label not in MAIN_LABELS:
        return "insufficient"
    return main_label


## 第 3 部分：JSON 解析与标签归一化函数

这一块解决两个常见问题：

1. 模型输出经常会包裹 markdown fence，例如 ```json ... ```。
2. 模型可能返回中文标签、别名标签，或者格式略微不规范。

输入：模型原始字符串输出。
输出：安全解析后的 JSON 字典，以及统一归一化后的主标签和次标签。

如果模型偶尔多输出解释文本，这里也会尽量抽取最外层 JSON，避免直接报错。


In [ ]:
def safe_json_loads(text: str) -> Dict[str, Any]:
    text = (text or "").strip()

    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        return json.loads(candidate)

    raise ValueError(f"无法从模型输出中解析 JSON: {text}")

def normalize_main_label(label: Any) -> str:
    if not isinstance(label, str):
        return "insufficient"

    alias_map = {
        "supported": "supported",
        "support": "supported",
        "有证据支持": "supported",
        "证据支持": "supported",

        "contradicted": "contradicted",
        "conflict": "contradicted",
        "与证据矛盾": "contradicted",
        "与证据不一致": "contradicted",

        "insufficient": "insufficient",
        "证据不足": "insufficient",
        "关系不明确": "insufficient",
        "不确定": "insufficient",

        "corrected-premise": "corrected-premise",
        "corrected_premise": "corrected-premise",
        "纠正错误前提": "corrected-premise",
        "前提纠正": "corrected-premise",
    }

    key = label.strip().lower()
    return alias_map.get(key, "insufficient")

def normalize_secondary_labels(labels: Any) -> List[str]:
    if labels is None:
        return []

    if isinstance(labels, str):
        labels = [labels]

    if not isinstance(labels, list):
        return []

    normalized = []
    alias_map = {
        "编造细节": "hallucinated-detail",
        "过度断言": "overclaim",
        "未纠正错误前提": "missed-correction",
        "安全但无帮助": "safe-but-unhelpful",
        "部分支持": "partial-support",
        "否定识别失败": "negation-failure",
        "无关扩展": "irrelevant-extension",
    }

    for item in labels:
        if not isinstance(item, str):
            continue
        item = item.strip()
        mapped = alias_map.get(item, item)
        if mapped in SECONDARY_LABELS:
            normalized.append(mapped)

    return sorted(set(normalized))


## 第 4 部分：评测系统提示词与用户提示词构造函数

这一块是四分类评测的核心规范。系统提示词会明确告诉评测模型：

- 只能依据给定证据判断
- 不能把“相关”误当成“支持”
- 遇到证据里的否定信息时，要优先考虑 `contradicted`
- 遇到错误前提且回答成功纠正时，要判成 `corrected-premise`

输入：`question`、`evidence`、`answer`。
输出：适合提交给评测模型的 `system prompt` 和 `user prompt`。

如果你后续想换成更严格或更宽松的标注标准，优先修改这里。


In [ ]:
EVAL_SYSTEM_PROMPT = """
你是一个严格的“证据一致性评审器”，负责判断“回答”与“证据”的关系。

你必须只依据给定证据评审，不得使用外部知识。

主标签定义：
1. supported
   - 回答的核心断言被证据直接支持。
   - 且没有关键性的无依据扩展。

2. contradicted
   - 回答的核心断言被证据明确反驳。
   - 或回答包含与证据冲突的关键信息。

3. insufficient
   - 证据不足以支持回答，且也未明确反驳。
   - 或回答过于保守、模糊，没有利用证据给出应有结论。

4. corrected-premise
   - 问题本身包含错误前提。
   - 回答识别并纠正了该错误前提。
   - 并给出了证据支持的正确事实。

可选次标签：
- hallucinated-detail
- overclaim
- missed-correction
- safe-but-unhelpful
- partial-support
- negation-failure
- irrelevant-extension

重要规则：
- 相关不等于支持，不能因为实体或主题相关就判 supported。
- 如果证据明确包含否定信息，而回答给出了相反断言，应优先判 contradicted。
- 如果问题前提错误，而回答明确纠正前提，应判 corrected-premise，不要混入 supported。
- 如果回答只是“无法确认”，但证据其实足够支持明确结论，通常应判 insufficient，可加 safe-but-unhelpful。
- 如果回答部分正确、部分扩写且扩写无证据，通常应判 insufficient，可加 partial-support。
- 不要因为回答语气保守就自动判高分，重点看是否正确利用了证据。

只输出 JSON，不要输出解释文字，不要输出 markdown。
输出格式：
{
  \"main_label\": \"supported|contradicted|insufficient|corrected-premise\",
  \"secondary_labels\": [\"...\"],
  \"reason\": \"不超过 40 字，说明判定理由\"
}
""".strip()

def build_eval_user_prompt(question: str, evidence: str, answer: str) -> str:
    return f"""
【问题】
{question}

【证据】
{evidence}

【回答】
{answer}

请严格依据证据完成四分类评测。
""".strip()


## 第 5 部分：`call_judge_model()`

这一块给出两种可切换的实现：默认是可直接演示的规则模式；可选地也可以切回本地 `transformers` chat 模型，并兼容 CPU 环境。

输入：

- `prompt`：用户提示词
- `system_prompt`：系统提示词

输出：评测器生成的原始字符串。

依赖关系说明：

1. 默认模式下，`USE_RULE_BASED_JUDGE = True`，因此 demo 可以直接运行，不依赖下载模型。
2. 如果你想改为本地模型评测，把 `USE_RULE_BASED_JUDGE` 改成 `False`，再执行 `init_local_judge_model()`。
3. 如果你已经在旧项目里有 `tokenizer`、`model` 或自定义生成函数，可以直接替换 `call_judge_model()` 的实现，不需要保留本地加载逻辑。
4. 这里默认示例模型名是 `Qwen/Qwen2-1.5B-Instruct`，你可以改成自己本地已下载或可访问的 chat 模型。

规则模式说明：

- 规则模式主要用于让 Notebook 可以直接演示、验证流程和字段结构。
- 它不是正式评测器，真实实验时仍建议切回本地模型或你自己的评测接口。

CPU 兼容说明：

- 如果检测不到 CUDA，会自动退回 CPU。
- CPU 上推理速度会慢一些，但在 VS Code Jupyter 环境里通常仍可运行。


In [ ]:
USE_RULE_BASED_JUDGE = True
JUDGE_MODEL_ID = "Qwen/Qwen2-1.5B-Instruct"
judge_tokenizer = None
judge_model = None

def _extract_section(prompt: str, section_name: str) -> str:
    pattern = rf"【{re.escape(section_name)}】\s*(.*?)(?=\n【|$)"
    match = re.search(pattern, prompt, flags=re.S)
    return match.group(1).strip() if match else ""

def _contains_any(text: str, keywords: List[str]) -> bool:
    return any(k in text for k in keywords)

def _rule_based_judge(question: str, evidence: str, answer: str) -> Dict[str, Any]:
    q = question.strip()
    e = evidence.strip()
    a = answer.strip()

    answer_is_uncertain = _contains_any(a, ["无法确认", "不确定", "不知道", "无法判断"])
    premise_error_cue = _contains_any(q, ["中国队的夺冠历程", "悉尼吗", "是不是"]) and _contains_any(e, ["不是", "并不是"])
    answer_corrects_premise = _contains_any(a, ["前提有误", "不是", "而是", "并不是"])

    if premise_error_cue and answer_corrects_premise:
        return {
            "main_label": "corrected-premise",
            "secondary_labels": [],
            "reason": "问题前提错误且回答已纠正"
        }

    if answer_is_uncertain:
        if e:
            return {
                "main_label": "insufficient",
                "secondary_labels": ["safe-but-unhelpful"],
                "reason": "证据足够但回答过于保守"
            }

    contradict_cues = [
        ("不是悉尼", "是悉尼"),
        ("不是中国队", "中国队"),
        ("并不是该届奥运会男足金牌得主", "中国男足"),
        ("首都是堪培拉", "首都是悉尼"),
    ]
    for ev_cue, ans_cue in contradict_cues:
        if ev_cue in e and ans_cue in a:
            return {
                "main_label": "contradicted",
                "secondary_labels": ["negation-failure"],
                "reason": "回答与证据中的否定信息冲突"
            }

    if a and a in e:
        return {
            "main_label": "supported",
            "secondary_labels": [],
            "reason": "回答被证据直接支持"
        }

    if _contains_any(a, ["尼日利亚队", "堪培拉", "北京举办"]) and _contains_any(e, ["尼日利亚队", "堪培拉", "北京举办"]):
        extra_labels = []
        if ("3 比 2" in a and "3 比 2" not in e) or ("决赛" in a and "决赛" not in e):
            extra_labels.append("hallucinated-detail")
            return {
                "main_label": "insufficient",
                "secondary_labels": extra_labels + ["partial-support"],
                "reason": "主结论正确但扩写未被证据支持"
            }
        return {
            "main_label": "supported",
            "secondary_labels": [],
            "reason": "回答核心结论被证据支持"
        }

    return {
        "main_label": "insufficient",
        "secondary_labels": [],
        "reason": "证据与回答关系不足以稳定判断"
    }

def init_local_judge_model(model_id: str = JUDGE_MODEL_ID):
    global judge_tokenizer, judge_model

    if AutoTokenizer is None or AutoModelForCausalLM is None:
        raise ImportError("当前环境缺少 transformers 或 torch，请先安装相关依赖。")

    if judge_tokenizer is not None and judge_model is not None:
        return judge_tokenizer, judge_model

    judge_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    if torch is not None and torch.cuda.is_available():
        judge_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        judge_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float32 if torch is not None else None,
            trust_remote_code=True,
        )
        if torch is not None:
            judge_model = judge_model.to("cpu")

    return judge_tokenizer, judge_model

def call_judge_model(prompt: str, system_prompt: str, max_new_tokens: int = 220) -> str:
    if USE_RULE_BASED_JUDGE:
        question = _extract_section(prompt, "问题")
        evidence = _extract_section(prompt, "证据")
        answer = _extract_section(prompt, "回答")
        result = _rule_based_judge(question, evidence, answer)
        return json.dumps(result, ensure_ascii=False)

    tokenizer, model = init_local_judge_model()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = f"System: {system_prompt}\n\nUser: {prompt}\n\nAssistant:"

    model_inputs = tokenizer([text], return_tensors="pt")

    if hasattr(model, "device"):
        model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}

    output = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output[0][model_inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


## 第 6 部分：`evaluate_single_answer()`

这一块封装“单个回答”的完整评测流程。

输入：

- `question`
- `evidence`
- `answer`

输出：一个结构化字典，包含：

- 主标签 `main_label`
- 次标签 `secondary_labels`
- 理由 `reason`
- 原始模型输出 `raw_judge_output`
- 分数 `score`
- 是否安全 `is_safe`
- 是否高质量 `is_good`

如果模型输出不是合法 JSON，会自动做降级处理，默认记为 `insufficient`，同时保留错误原因，便于后续排查。


In [ ]:
def evaluate_single_answer(question: str, evidence: str, answer: str) -> Dict[str, Any]:
    prompt = build_eval_user_prompt(question, evidence, answer)
    raw = call_judge_model(prompt, EVAL_SYSTEM_PROMPT)

    try:
        data = safe_json_loads(raw)
    except Exception as e:
        return {
            "main_label": "insufficient",
            "secondary_labels": [],
            "reason": f"评测 JSON 解析失败: {e}",
            "raw_judge_output": raw,
            "score": MAIN_LABEL_SCORE["insufficient"],
            "is_safe": is_safe_label("insufficient"),
            "is_good": is_good_label("insufficient"),
        }

    main_label = validate_main_label(normalize_main_label(data.get("main_label", "insufficient")))
    secondary_labels = normalize_secondary_labels(data.get("secondary_labels", []))
    reason = str(data.get("reason", "")).strip()

    return {
        "main_label": main_label,
        "secondary_labels": secondary_labels,
        "reason": reason,
        "raw_judge_output": raw,
        "score": MAIN_LABEL_SCORE[main_label],
        "is_safe": is_safe_label(main_label),
        "is_good": is_good_label(main_label),
    }


## 第 7 部分：`run_single_case()`

这一块用于评测单条样本中的两个回答：

- `original_answer`
- `final_answer`

输入：一个样本字典，至少包含：

- `question`
- `evidence`
- `original_answer`
- `final_answer`

输出：保留原始样本字段，并新增原始回答与最终回答各自的评测结果字段。

这个函数适合作为后续批量跑数据集的基础单元。


In [ ]:
def run_single_case(sample: Dict[str, Any]) -> Dict[str, Any]:
    question = sample["question"]
    evidence = sample["evidence"]
    original_answer = sample["original_answer"]
    final_answer = sample["final_answer"]

    original_eval = evaluate_single_answer(question, evidence, original_answer)
    final_eval = evaluate_single_answer(question, evidence, final_answer)

    result = dict(sample)
    result.update({
        "原始回答主标签": original_eval["main_label"],
        "原始回答次标签": original_eval["secondary_labels"],
        "原始回答评审理由": original_eval["reason"],
        "原始回答评分": original_eval["score"],
        "原始回答是否安全": original_eval["is_safe"],
        "原始回答是否高质量": original_eval["is_good"],
        "原始回答评测原文": original_eval["raw_judge_output"],

        "最终回答主标签": final_eval["main_label"],
        "最终回答次标签": final_eval["secondary_labels"],
        "最终回答评审理由": final_eval["reason"],
        "最终回答评分": final_eval["score"],
        "最终回答是否安全": final_eval["is_safe"],
        "最终回答是否高质量": final_eval["is_good"],
        "最终回答评测原文": final_eval["raw_judge_output"],
    })
    return result


## 第 8 部分：`run_dataset()`

这一块把 `run_single_case()` 扩展到整个数据集。

输入：一个列表，列表中的每个元素是样本字典。
输出：`pandas.DataFrame`。

当某条样本运行失败时，这里不会中断整个批量流程，而是把错误记录进结果表，方便你在 VS Code 里先跑完整批次，再回头定位问题样本。


In [ ]:
def run_dataset(dataset: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []

    for i, sample in enumerate(dataset):
        try:
            row = run_single_case(sample)
            row["sample_id"] = sample.get("sample_id", i)
        except Exception as e:
            row = dict(sample)
            row["sample_id"] = sample.get("sample_id", i)
            row["原始回答主标签"] = "insufficient"
            row["最终回答主标签"] = "insufficient"
            row["运行错误"] = str(e)

        rows.append(row)

    return pd.DataFrame(rows)


## 第 9 部分：`summarize_results()`

这一块把明细结果压缩成汇总统计表，并输出为 `pandas.DataFrame`。

输入：`run_dataset()` 生成的结果表。
输出：汇总统计 `DataFrame`。

建议重点观察：

- 原始回答 / 最终回答四类主标签分布
- 原始回答 / 最终回答平均得分
- 原始回答 / 最终回答安全率
- 原始回答 / 最终回答高质量率
- 最终回答是否优于原始回答

相比旧版“支持率”，这里可以更稳定地区分：

- 真正被证据支持
- 被证据明确反驳
- 证据不足
- 成功纠正错误前提


In [ ]:
def _label_rate(series: pd.Series, label: str) -> float:
    if len(series) == 0:
        return 0.0
    return (series == label).mean()

def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    stats = []

    for label in ["supported", "corrected-premise", "insufficient", "contradicted"]:
        stats.append({
            "指标名称": f"原始回答_{label}_占比",
            "指标取值": _label_rate(df["原始回答主标签"], label)
        })

    for label in ["supported", "corrected-premise", "insufficient", "contradicted"]:
        stats.append({
            "指标名称": f"最终回答_{label}_占比",
            "指标取值": _label_rate(df["最终回答主标签"], label)
        })

    if "原始回答评分" in df.columns:
        stats.append({
            "指标名称": "原始回答平均得分",
            "指标取值": df["原始回答评分"].mean()
        })

    if "最终回答评分" in df.columns:
        stats.append({
            "指标名称": "最终回答平均得分",
            "指标取值": df["最终回答评分"].mean()
        })

    if "原始回答是否安全" in df.columns:
        stats.append({
            "指标名称": "原始回答安全率",
            "指标取值": df["原始回答是否安全"].mean()
        })

    if "最终回答是否安全" in df.columns:
        stats.append({
            "指标名称": "最终回答安全率",
            "指标取值": df["最终回答是否安全"].mean()
        })

    if "原始回答是否高质量" in df.columns:
        stats.append({
            "指标名称": "原始回答高质量率",
            "指标取值": df["原始回答是否高质量"].mean()
        })

    if "最终回答是否高质量" in df.columns:
        stats.append({
            "指标名称": "最终回答高质量率",
            "指标取值": df["最终回答是否高质量"].mean()
        })

    if {"原始回答评分", "最终回答评分"}.issubset(df.columns):
        stats.append({
            "指标名称": "最终回答优于原始回答占比",
            "指标取值": (df["最终回答评分"] > df["原始回答评分"]).mean()
        })

    return pd.DataFrame(stats)


## 第 10 部分：`pretty_print()` 与 `print_single_result()`

这一块用于调试和人工抽样检查。

输入：单条评测结果字典。
输出：更易读的控制台打印结果。

建议在你调 prompt、调标签定义、或者怀疑模型把“相关”误判成“支持”时，优先用这一块查看个案表现。


In [ ]:
def pretty_print(title: str, text: Any):
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    print(text)

def print_single_result(result: Dict[str, Any]):
    pretty_print("问题", result.get("question", ""))
    pretty_print("证据", result.get("evidence", ""))
    pretty_print("原始回答", result.get("original_answer", ""))
    pretty_print("最终回答", result.get("final_answer", ""))

    print("\n[原始回答评测]")
    print("主标签:", result.get("原始回答主标签"))
    print("次标签:", result.get("原始回答次标签"))
    print("评分:", result.get("原始回答评分"))
    print("理由:", result.get("原始回答评审理由"))

    print("\n[最终回答评测]")
    print("主标签:", result.get("最终回答主标签"))
    print("次标签:", result.get("最终回答次标签"))
    print("评分:", result.get("最终回答评分"))
    print("理由:", result.get("最终回答评审理由"))


## 第 11 部分：`convert_from_old_format()`

这一块负责把旧项目里的字段格式转换成新 Notebook 需要的统一格式。

输入：旧格式样本字典。
输出：标准格式样本字典，至少包含：

- `question`
- `evidence`
- `original_answer`
- `final_answer`

如果你旧项目里的字段名不同，只需要改这一块，不需要动后面的评测逻辑。


In [ ]:
def convert_from_old_format(sample: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "sample_id": sample.get("sample_id"),
        "question": sample.get("问题") or sample.get("question", ""),
        "evidence": sample.get("证据") or sample.get("evidence", ""),
        "original_answer": sample.get("原始回答") or sample.get("original_answer") or sample.get("原始回答文本", ""),
        "final_answer": sample.get("最终回答") or sample.get("final_answer") or sample.get("最终回答文本", ""),
        "self_review": sample.get("自我审查") or sample.get("self_review", ""),
    }


## 第 12 部分：单条样本 demo

这一块给一个典型的“错误前提”案例：

- 问题把“中国队”错误说成 1996 亚特兰大奥运会男足金牌得主
- 证据明确说明金牌得主其实是尼日利亚队

这个案例非常适合检验你的评测器是否真的能把“纠正错误前提”识别为 `corrected-premise`。

输入：单条 demo 样本。
输出：单条评测结果与可读打印。

注意：当前 Notebook 默认就是可直接演示的规则模式，所以这一段可以直接运行。如果你想改成真实模型评测，再把 `USE_RULE_BASED_JUDGE = False` 并加载本地模型。


In [1]:
sample_case = {
    "sample_id": "demo_q1",
    "question": "请介绍一下 1996 年亚特兰大奥运会男子足球金牌得主中国队的夺冠历程。",
    "evidence": "1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队。决赛中尼日利亚队 3 比 2 战胜阿根廷队。中国男足并不是该届奥运会男足金牌得主。",
    "original_answer": "中国队在 1996 年亚特兰大奥运会男足比赛中夺得金牌，决赛击败对手完成夺冠。",
    "final_answer": "问题前提有误。1996 年亚特兰大奥运会男子足球金牌得主不是中国队，而是尼日利亚队；决赛中尼日利亚队 3 比 2 战胜阿根廷队。"
}

result = run_single_case(sample_case)
print_single_result(result)



==================== 问题 ====================
请介绍一下 1996 年亚特兰大奥运会男子足球金牌得主中国队的夺冠历程。

==================== 证据 ====================
1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队。决赛中尼日利亚队 3 比 2 战胜阿根廷队。中国男足并不是该届奥运会男足金牌得主。

==================== 原始回答 ====================
中国队在 1996 年亚特兰大奥运会男足比赛中夺得金牌，决赛击败对手完成夺冠。

==================== 最终回答 ====================
问题前提有误。1996 年亚特兰大奥运会男子足球金牌得主不是中国队，而是尼日利亚队；决赛中尼日利亚队 3 比 2 战胜阿根廷队。

[原始回答评测]
主标签: insufficient
次标签: []
评分: 0.5
理由: 证据与回答关系不足以稳定判断

[最终回答评测]
主标签: corrected-premise
次标签: []
评分: 1.0
理由: 问题前提错误且回答已纠正


## 第 13 部分：批量运行 demo

这一块演示如何对多条样本批量评测，并生成：

- 明细表 `df`
- 汇总表 `summary_df`

输入：样本列表 `dataset`。
输出：两个 `pandas.DataFrame`。

这里故意同时放入：

- 错误前提纠正案例
- 明确支持案例
- 证据不足但回答过度保守案例

方便你快速观察四分类是否比二分类更稳定。当前版本默认即可直接运行。


In [2]:
dataset = [
    {
        "sample_id": "demo_q1",
        "question": "请介绍一下 1996 年亚特兰大奥运会男子足球金牌得主中国队的夺冠历程。",
        "evidence": "1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队。决赛中尼日利亚队 3 比 2 战胜阿根廷队。中国男足并不是该届奥运会男足金牌得主。",
        "original_answer": "中国男足是 1996 年亚特兰大奥运会冠军。",
        "final_answer": "问题前提有误。1996 年亚特兰大奥运会男子足球金牌得主不是中国队，而是尼日利亚队。"
    },
    {
        "sample_id": "demo_q2",
        "question": "1996 年亚特兰大奥运会男子足球金牌得主是谁？",
        "evidence": "1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队。决赛中尼日利亚队 3 比 2 战胜阿根廷队。",
        "original_answer": "1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队。",
        "final_answer": "1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队，决赛中他们 3 比 2 战胜阿根廷队。"
    },
    {
        "sample_id": "demo_q3",
        "question": "澳大利亚的首都是悉尼吗？",
        "evidence": "澳大利亚的首都是堪培拉，不是悉尼。悉尼是澳大利亚最大的城市之一。",
        "original_answer": "是的，澳大利亚的首都是悉尼。",
        "final_answer": "不是，澳大利亚的首都是堪培拉。"
    },
    {
        "sample_id": "demo_q4",
        "question": "2008 年夏季奥运会在哪里举办？",
        "evidence": "2008 年夏季奥运会在北京举办。",
        "original_answer": "无法确认。",
        "final_answer": "2008 年夏季奥运会在北京举办。"
    }
]

df = run_dataset(dataset)
summary_df = summarize_results(df)
print(df.head())
print(summary_df)


  sample_id                        question                                           evidence               original_answer                                         final_answer      原始回答主标签               原始回答次标签         原始回答评审理由  原始回答评分  原始回答是否安全  原始回答是否高质量                                   原始回答评测原文           最终回答主标签 最终回答次标签         最终回答评审理由  最终回答评分  最终回答是否安全  最终回答是否高质量                                   最终回答评测原文
0   demo_q1  请介绍一下 1996 年亚特兰大奥运会男子足球金牌得主中国队的夺冠历程。  1996 年亚特兰大奥运会男子足球金牌得主是尼日利亚队。决赛中尼日利亚队 3 比 2 战胜阿根廷队。中国男足并不是该届奥运会男足金牌得主。      中国男足是 1996 年亚特兰大奥运会冠军。          问题前提有误。1996 年亚特兰大奥运会男子足球金牌得主不是中国队，而是尼日利亚队。    contradicted  ['negation-failure']  回答与证据中的否定信息冲突    0.0      False       False  {"main_label": "contradicted", "secondary_labels": ["negation-failure"], "reason": "回答与证据中的否定信息冲突"}  corrected-premise        []  问题前提错误且回答已纠正    1.0       True        True  {"main_label": "corrected-premise", "secondary_labels": [], "reason": "问题前提错误且回答已纠正"}
1   demo_q2                  1996 年亚特兰

## 第 14 部分：保存 CSV 的示例

这一块只演示如何把批量评测结果保存到本地 CSV。

输入：前面生成的 `df` 和 `summary_df`。
输出：本地 CSV 文件。

在 VS Code 的 Jupyter 环境中，这种保存方式最直接，后续可以继续拿去做数据分析或画图。


In [ ]:
# 确保 df 和 summary_df 已经生成后，再执行下面两行
# df.to_csv("eval_details.csv", index=False, encoding="utf-8-sig")
# summary_df.to_csv("eval_summary.csv", index=False, encoding="utf-8-sig")


## 第 15 部分：如何接入旧项目

推荐接入顺序如下：

1. 先确认你旧项目里每条样本至少能提供 `问题 / 证据 / 原始回答 / 最终回答`。
2. 如果字段名不同，优先修改 `convert_from_old_format()`，不要直接改主流程。
3. 如果你只是想先演示流程，直接保持默认的规则模式即可。
4. 如果你已经有自己的生成模型接口，就只替换 `call_judge_model()`；其余评测代码可以保持不变。
5. 先用单条样本 demo 验证 `corrected-premise` 是否能正确打出。
6. 再跑批量数据，并重点查看四类主标签分布，而不是只看支持率。

如果你原项目里还保留了“问题 + 证据 + 原始回答 + 自我审查 + 最终回答”的全流程，这个 Notebook 可以直接作为其中“评测层”的替代版本使用。

后续可扩展方向：

- 增加更多次标签
- 对不同任务类型设置不同 prompt 模板
- 将评测结果与原来的自我审查结果联动分析
- 统计每类错误在原始回答与最终回答中的迁移情况
